In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [2]:
%load_ext cudf.pandas

/opt/conda/envs/rapids-25.02/lib/python3.10/site-packages/cudf/pandas/__init__.py:65: UserWarning: cudf.pandas detected an already configured memory resource, ignoring 'CUDF_PANDAS_RMM_MODE'=managed_pool
  warnings.warn(


In [3]:
%LoadCheckpoint /home/dias-benchmarks/tpch/notebooks/q05/annotated/checkpoints/post_cell_3.pickle

trying: ['orders']


me:  3
trying: ['orig_output']
me:  2
trying: ['nation']
me:  7
trying: ['pd']
me:  0
trying: ['tpch_parent']
me:  0
trying: ['STORAGE_OPTS']
me:  0
trying: ['lineitem']


me:  1
trying: ['customer']
me:  5
trying: ['load_lineitem']
me:  1
trying: ['DATA_ROOT']
me:  0
trying: ['load_nation']
me:  7
trying: ['load_orders']
me:  3
trying: ['load_customer']
me:  5


Declaring variable pd
Declaring variable tpch_parent
Declaring variable STORAGE_OPTS
Declaring variable DATA_ROOT
Declaring variable lineitem
Declaring variable load_lineitem
Declaring variable orig_output
Declaring variable orders
Declaring variable load_orders
Declaring variable customer
Declaring variable load_customer
Declaring variable nation
Declaring variable load_nation


In [4]:
%%RecordEvent
%%time
### cell 4 ###

def load_region(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = f"{data_folder}/region.tbl"
    # By specifying names, usecols, and dtypes up front we eliminate extra inference passes
    col_names = ["r_regionkey", "r_name", "r_comment"]
    df = pd.read_csv(
        data_path,
        sep="|",
        header=None,
        names=col_names + ["_tmp"],  # _tmp to swallow the trailing delimiter
        usecols=col_names,
        dtype={
            "r_regionkey": "int32",
            "r_name": "str",
            "r_comment": "str"
        }
    )
    return df

region = load_region(DATA_ROOT, **STORAGE_OPTS)

CPU times: user 8.42 ms, sys: 52.3 ms, total: 60.7 ms
Wall time: 63.1 ms


In [5]:
%Checkpoint /home/dias-benchmarks/tpch/notebooks/q05/rewritten/o4_mini_high_q5/checkpoints/post_cell_4_try_1.pickle

migration speed (bps): 793269051.6022102
---------------------------
variables to migrate:
DATA_ROOT 80
STORAGE_OPTS 64
orders 48
load_region 144
load_orders 144
pd 72
orig_output 16
nation 48
load_customer 144
load_nation 144
customer 48
tpch_parent 54
load_lineitem 144
region 48
lineitem 48
---------------------------
variables to recompute:
[]
---------------------------
cells to recompute:
[]
Checkpoint saved to: /home/dias-benchmarks/tpch/notebooks/q05/rewritten/o4_mini_high_q5/checkpoints/post_cell_4_try_1.pickle


In [6]:
%PrintCellInfo opt_cell_exec_info

======= Cell 0 =======
Input variables []
Active variables []
Intermediate variables ['lineitem']
Future variables []
Modified dataframes
======= Cell 1 =======
Input variables []
Active variables []
Intermediate variables ['orders']
Future variables []
Modified dataframes
======= Cell 2 =======
Input variables []
Active variables []
Intermediate variables ['customer']
Future variables []
Modified dataframes
======= Cell 3 =======
Input variables []
Active variables []
Intermediate variables ['nation']
Future variables []
Modified dataframes
======= Cell 4 =======
Input variables []
Active variables []
Intermediate variables ['region']
Future variables []
Modified dataframes
Saved cell execution info to opt_cell_exec_info


In [7]:

with open("/home/dias-benchmarks/tpch/notebooks/q05/opt_cell_exec_info_4_try_1.pkl", "wb") as f:
    pickle.dump(opt_cell_exec_info[4], f)


In [ ]:
opt_output = Out.get(4)

In [ ]:
%LoadCheckpoint /home/dias-benchmarks/tpch/notebooks/q05/annotated/checkpoints/post_cell_4.pickle

import numpy as np
import cudf
from elastic.core.common.pandas import is_type_styler
is_orig_output_pd = isinstance(orig_output, (pd.Series, pd.DataFrame, pd.Index))
is_opt_output_pd = isinstance(opt_output, (pd.Series, pd.DataFrame, pd.Index))
is_orig_output_array = isinstance(orig_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray))
is_opt_output_array = isinstance(opt_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray))
is_orig_output_styler = is_type_styler(type(orig_output))
is_opt_output_styler = is_type_styler(type(opt_output))
if is_orig_output_styler and is_opt_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_orig_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_opt_output_styler:
    assert opt_output.to_html() == orig_output

if is_orig_output_pd and is_opt_output_pd:
    assert orig_output.equals(opt_output)
# TODO(jie): this is a hack.
elif ((is_orig_output_pd or is_opt_output_pd) and (is_orig_output_array or is_opt_output_array)) or (is_orig_output_array and is_opt_output_array):
    assert list(orig_output) == list(opt_output)
else:
    assert orig_output == opt_output
